## Structured Output

Structured output allows an LLM to return responses in a predefined format instead of plain text.

Using structured output, we can define a schema for the expected response, such as specific fields, data types, or a Pydantic model. LangChain then ensures that the model's response follows this structure.

This is useful when the output needs to be reliably consumed by applications, APIs, databases, or other program components.

### Example

Instead of:

"Python is a programming language created by Guido van Rossum."

We can request:

```python
{
    "name": "Python",
    "creator": "Guido van Rossum",
    "type": "Programming Language"
}

### Pydantic

Pydantic is a Python library used to define and validate structured data using Python type hints.

In LangChain, Pydantic models are commonly used with **structured output** to define the exact schema the LLM should return.

### Example

```python
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
    profession: str

In [13]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
    )
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EEBA2BB110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EEBA2BBB10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This year movie was released")
    rating:float=Field(description="Rating of this movie")
    deirector:str=Field(description="The direstor of this movie")


In [15]:
model_structured_output = model.with_structured_output(Movie)
model_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EEBA2BB110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EEBA2BBB10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'para

In [16]:
model.invoke("Provide me the details about movie inception")

AIMessage(content='**Inception**  \n*(2010 American science‑fiction action‑thriller)*  \n\n| Category | Details |\n|----------|---------|\n| **Director** | Christopher Nolan |\n| **Screenplay** | Christopher Nolan |\n| **Producers** | Christopher Nolan, Emma Thomas |\n| **Story** | Christopher Nolan |\n| **Music** | Hans Zimmer |\n| **Cinematography** | Wally Pfister |\n| **Editing** | Lee Smith |\n| **Production companies** | Syncopy, Warner Bros., Legendary Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Release dates** | –\u202fUnited Kingdom: 30\u202fJul\u202f2010 (theatrical) <br>–\u202fUnited States: 16\u202fJul\u202f2010 (theatrical) |\n| **Runtime** | 148 minutes |\n| **Country** | United States |\n| **Language** | English |\n| **Budget** | $160\u202fmillion |\n| **Box office** | $829\u202fmillion worldwide |\n| **Cast** | • **Leonardo DiCaprio** – Dom\u202fCobb <br>• **Joseph Gordon‑Levitt** – Arthur <br>• **Ellen Page** – Ariadne <br>• **Tom Hardy** – Eames <br>•

In [17]:
response = model_structured_output.invoke("Provide me details about the movie inception")
response

Movie(title='Inception', year=2010, rating=8.8, deirector='Christopher Nolan')

### Message output alongside parsed structure

In [18]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This year movie was released")
    rating:float=Field(description="Rating of this movie")
    deirector:str=Field(description="The direstor of this movie")

model_structured_output = model.with_structured_output(Movie, include_raw=True)
response = model_structured_output.invoke("Provide me details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide me details about the movie inception". We need to use the function "Movie" to get details about the movie inception. We should call the function with director, rating, title, year. We need to provide details about the movie. The function likely returns details. Let\'s call it.', 'tool_calls': [{'id': 'fc_cc033ef2-231e-40c6-8c24-912d0e9ff55b', 'function': {'arguments': '{"deirector":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 156, 'total_tokens': 261, 'completion_time': 0.108104486, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.007457004, 'prompt_tokens_details': None, 'queue_time': 0.157158236, 'total_time': 0.11556149}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'fi

### Nested Structure

In [19]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in Milloins USD")

model_structured_output = model.with_structured_output(MovieDetails)

response = model_structured_output.invoke("Provide me details about movie Inception.")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160000000.0)


### TypeDict
**In one sentence:** `TypedDict` defines **what keys and data types a dictionary should contain**, without providing Pydantic-style runtime validation.

In [20]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """A Movie with Details"""
    title: Annotated[str, ..., "The title of the Movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "The director of the Movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]


model_type_dict = model.with_structured_output(MovieDict)
response = model_type_dict.invoke("Provide details about movie Avengers.")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2008}

In [21]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None # =  Field(None, description="Budget in Milloins USD")

model_structured_output = model.with_structured_output(MovieDetails)

response = model_structured_output.invoke("Provide me details about movie Inception.")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Marion Cotillard', 'role': 'Mal'}],
 'genres': ['Action', 'Science Fiction', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

### DataClasses
A data class is typically containing mainly data, although there are not any restrictions. We create it using **@dataclass** decotrator.

In [26]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="Name of person")
    email: str = Field(description="The email of a person")
    phone: str = Field(description="The phone number of person")

llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
)
agent  = create_agent(
    model = llm,
    response_format= ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

result

name='John Doe' email='john@example.com' phone='(555) 123-4567'


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6c5355a9-039f-4786-9c43-ae27436a11c3'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON matching schema ContactInfo. Must be compact JSON. Ensure all required fields: name, email, phone. Provide values from input. Name: "John Doe". Email: "john@example.com". Phone: "(555) 123-4567". Ensure compact JSON. Output only JSON object.'}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 234, 'total_tokens': 335, 'completion_time': 0.116774415, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.011307027, 'prompt_tokens_details': None, 'queue_time': 0.197279039, 'total_time': 0.128081442}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_

In [28]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
# Using TypeDict

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str 
    email: str 
    phone: str 

llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
)
agent  = create_agent(
    model = llm,
    response_format= ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [32]:
# Using DataClasses
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str 
    email: str 
    phone: str 

agent  = create_agent(
    model = llm,
    response_format= ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')